# 🫀 Phase 3 — Cardiac Disease Classifier GUI
---

| Phase | Topic |
|-------|-------|
| 1 | Synthetic Dataset Generator |
| 2 | EDA, Modelling and Evaluation |
| **3** | **GUI using Gradio ← You are here** |
| 4 | Report |

---

### What this notebook does
1. **Installs** all required packages automatically
2. **Re-trains** the model from scratch if saved artefacts are missing
3. **Launches** a fully interactive Gradio web-app that:
   - Accepts 16 clinical inputs (including **height + weight → BMI auto-calculator**)
   - Classifies the patient into one of four classes
   - Shows probability bars, confidence badge, and clinical advice
   - Provides a **public shareable link** (Gradio share=True)

---

### 🚀 How to run
Click **Run All** (or press `Shift+Enter` on each cell in order).  
The last cell will print a local URL **and** a public URL you can share.


## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = ["gradio>=4.0.0", "scikit-learn>=1.3.0",
        "numpy>=1.24.0", "pandas>=2.0.0", "scipy>=1.11.0"]

for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages ready.")

## Step 2 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import pathlib
import warnings
import gradio as gr
from scipy.stats import truncnorm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings('ignore')
np.random.seed(42)

print(f"gradio  : {gr.__version__}")
print("✅ Imports complete.")

## Step 3 — Train or Load Model

If `model.pkl` already exists (from Phase 2), it is loaded directly.  
Otherwise the training pipeline runs here automatically so Phase 3 is self-contained.

In [ ]:
# ─── Constants ────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['Healthy', 'CAD', 'Arrhythmia', 'Heart Failure']
CONT_COLS    = ['age', 'bmi', 'systolic_bp', 'ldl', 'fasting_glucose',
                'heart_rate', 'spo2', 'ejection_fraction', 'bnp']
CAT_COLS     = ['sob', 'chest_tightness', 'smoking', 'diabetes',
                'edema', 'palpitations', 'ecg']
FEAT_COLS    = CONT_COLS + CAT_COLS
N = 600

# ─── Sampling helpers ─────────────────────────────────────────────────────────
def tnorm(mean, sd, lo, hi, n=N):
    a, b = (lo-mean)/sd, (hi-mean)/sd
    return truncnorm.rvs(a, b, loc=mean, scale=sd, size=n)

def lognorm_trunc(mu_log, sd_log, lo, hi, n=N):
    a = (np.log(lo)-mu_log)/sd_log
    b = (np.log(hi)-mu_log)/sd_log
    return np.exp(truncnorm.rvs(a, b, loc=mu_log, scale=sd_log, size=n))

def cat(vals, probs, n=N):
    return np.random.choice(vals, n, p=probs)

# ─── Per-class generators ─────────────────────────────────────────────────────
def gen_class(label):
    if label == 'Healthy':
        d = dict(
            age=tnorm(48,14,20,75),         bmi=tnorm(29,5,18.5,40),
            systolic_bp=tnorm(124,12,100,148), ldl=tnorm(128,34,60,195),
            fasting_glucose=tnorm(98,14,70,125), heart_rate=tnorm(78,11,55,100),
            spo2=tnorm(97,1.5,94,100),      ejection_fraction=tnorm(62,7,48,76),
            bnp=lognorm_trunc(3.45,0.92,5,200),
            sob=cat([0,1,2,3],[.80,.20,.00,.00]),
            chest_tightness=cat([0,1,2],[.87,.11,.02]),
            smoking=cat([0,1,2,3],[.65,.20,.12,.03]),
            diabetes=cat([0,1,2,3],[.66,.26,.07,.01]),
            edema=cat([0,1,2],[.93,.06,.01]),
            palpitations=cat([0,1,2],[.38,.55,.07]),
            ecg=cat([0,1,2,3],[.80,.18,.02,.00]))
    elif label == 'CAD':
        d = dict(
            age=tnorm(61,11,40,82),          bmi=tnorm(33,6,22,44),
            systolic_bp=tnorm(153,18,118,188), ldl=tnorm(175,45,85,265),
            fasting_glucose=tnorm(145,33,80,210), heart_rate=tnorm(80,16,48,112),
            spo2=tnorm(94,3,88,99),          ejection_fraction=tnorm(51,10,32,70),
            bnp=lognorm_trunc(5.23,0.57,60,580),
            sob=cat([0,1,2,3],[.28,.38,.24,.10]),
            chest_tightness=cat([0,1,2],[.20,.38,.42]),
            smoking=cat([0,1,2,3],[.44,.22,.22,.12]),
            diabetes=cat([0,1,2,3],[.42,.18,.30,.10]),
            edema=cat([0,1,2],[.64,.28,.08]),
            palpitations=cat([0,1,2],[.30,.50,.20]),
            ecg=cat([0,1,2,3],[.22,.42,.28,.08]))
    elif label == 'Arrhythmia':
        d = dict(
            age=tnorm(58,15,28,88),          bmi=tnorm(31,6,20,42),
            systolic_bp=tnorm(145,19,108,182), ldl=tnorm(147,39,68,225),
            fasting_glucose=tnorm(127,26,75,178), heart_rate=tnorm(102,32,38,165),
            spo2=tnorm(93,3,86,99),          ejection_fraction=tnorm(54,9,35,72),
            bnp=lognorm_trunc(4.70,0.85,20,600),
            sob=cat([0,1,2,3],[.30,.38,.24,.08]),
            chest_tightness=cat([0,1,2],[.62,.28,.10]),
            smoking=cat([0,1,2,3],[.54,.22,.18,.06]),
            diabetes=cat([0,1,2,3],[.58,.20,.18,.04]),
            edema=cat([0,1,2],[.74,.22,.04]),
            palpitations=cat([0,1,2],[.22,.40,.38]),
            ecg=cat([0,1,2,3],[.14,.38,.28,.20]))
    else:  # Heart Failure
        d = dict(
            age=tnorm(65,13,40,90),          bmi=tnorm(31,7,17,44),
            systolic_bp=tnorm(146,23,100,192), ldl=tnorm(134,43,48,220),
            fasting_glucose=tnorm(193,59,76,310), heart_rate=tnorm(89,22,46,132),
            spo2=tnorm(91,4,82,99),          ejection_fraction=tnorm(58,9,40,75),
            bnp=lognorm_trunc(5.79,1.11,35,3000),
            sob=cat([0,1,2,3],[.20,.30,.32,.18]),
            chest_tightness=cat([0,1,2],[.40,.42,.18]),
            smoking=cat([0,1,2,3],[.52,.22,.18,.08]),
            diabetes=cat([0,1,2,3],[.30,.14,.42,.14]),
            edema=cat([0,1,2],[.28,.38,.34]),
            palpitations=cat([0,1,2],[.32,.36,.32]),
            ecg=cat([0,1,2,3],[.10,.28,.32,.30]))
    t = pd.DataFrame(d)
    t['target'] = label
    return t

# ─── Load or Train ────────────────────────────────────────────────────────────
ARTEFACTS = ['model.pkl','scaler.pkl','label_encoder.pkl','feature_cols.pkl']

if all(pathlib.Path(a).exists() for a in ARTEFACTS):
    print("Loading saved artefacts from Phase 2 …")
    with open('model.pkl','rb')         as f: MODEL    = pickle.load(f)
    with open('scaler.pkl','rb')        as f: SCALER   = pickle.load(f)
    with open('label_encoder.pkl','rb') as f: LE       = pickle.load(f)
    with open('feature_cols.pkl','rb')  as f: FEAT_COLS= pickle.load(f)
    CAT_COLS = ['sob','chest_tightness','smoking','diabetes','edema','palpitations','ecg']
    print("✅ Model loaded.")
else:
    print("Artefacts not found — training from scratch …")
    df = pd.concat([gen_class(c) for c in CLASS_NAMES], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    for col in CONT_COLS:
        df[col] = df[col].clip(lower=0 if col not in ['bnp'] else 1)
    df['spo2'] = df['spo2'].clip(0,100)
    df['ejection_fraction'] = df['ejection_fraction'].clip(0,100)

    X = df.drop('target', axis=1)
    LE = LabelEncoder()
    y  = LE.fit_transform(df['target'])

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               random_state=42, stratify=y)
    cat_c = ['sob','chest_tightness','smoking','diabetes','edema','palpitations','ecg']
    num_c = [c for c in X.columns if c not in cat_c]

    SCALER = StandardScaler()
    Xtr = np.hstack([SCALER.fit_transform(X_tr[num_c]), X_tr[cat_c].values])
    Xte = np.hstack([SCALER.transform(X_te[num_c]),     X_te[cat_c].values])

    MODEL = RandomForestClassifier(n_estimators=200, max_depth=12,
                                   random_state=42, n_jobs=-1)
    MODEL.fit(Xtr, y_tr)
    acc = accuracy_score(y_te, MODEL.predict(Xte))
    print(f"Test accuracy: {acc:.3f}")
    print(classification_report(y_te, MODEL.predict(Xte), target_names=LE.classes_))

    FEAT_COLS = list(X.columns)
    CAT_COLS  = cat_c

    for fname, obj in [('model.pkl',MODEL),('scaler.pkl',SCALER),
                       ('label_encoder.pkl',LE),('feature_cols.pkl',FEAT_COLS)]:
        with open(fname,'wb') as f: pickle.dump(obj, f)
    print("✅ Model trained and saved.")

## Step 4 — Class Metadata & Prediction Function

In [ ]:
CLASS_COLORS = {
    'Healthy':       '#27ae60',
    'CAD':           '#e74c3c',
    'Arrhythmia':    '#f39c12',
    'Heart Failure': '#8e44ad',
}
CLASS_ICONS = {
    'Healthy':'💚', 'CAD':'❤️\u200d🔥',
    'Arrhythmia':'⚡', 'Heart Failure':'💜'
}
CLASS_DESC = {
    'Healthy':
        'No evidence of significant cardiac pathology. '
        'Maintain a heart-healthy lifestyle with regular check-ups.',
    'CAD':
        'Pattern consistent with Coronary Artery Disease. '
        'Lipid management, BP control and cardiology review advised.',
    'Arrhythmia':
        'Abnormal cardiac rhythm detected (AF / VT / Tachy-Brady). '
        'ECG confirmation and cardiology referral advised.',
    'Heart Failure':
        'Pattern consistent with Heart Failure — ventricular impairment. '
        'Urgent cardiology evaluation recommended.',
}

NUM_COLS = [c for c in FEAT_COLS if c not in CAT_COLS]

def predict(height_cm, weight_kg, bmi_input, bmi_mode,
            age, systolic_bp, ldl, fasting_glucose,
            heart_rate, spo2, ejection_fraction, bnp,
            sob, chest_tightness, smoking, diabetes,
            edema, palpitations, ecg):

    # BMI: use auto-calculated if height+weight provided, else use slider
    if bmi_mode == 'Calculate from Height & Weight' and height_cm > 0 and weight_kg > 0:
        bmi = weight_kg / ((height_cm / 100) ** 2)
    else:
        bmi = bmi_input

    bmi = round(float(bmi), 1)

    num_arr = np.array([[age, bmi, systolic_bp, ldl, fasting_glucose,
                         heart_rate, spo2, ejection_fraction, bnp]])
    cat_arr = np.array([[sob, chest_tightness, smoking, diabetes,
                         edema, palpitations, ecg]])
    X = np.hstack([SCALER.transform(num_arr), cat_arr])

    proba  = MODEL.predict_proba(X)[0]
    labels = LE.classes_
    order  = np.argsort(proba)[::-1]
    top    = labels[order[0]]
    top_p  = proba[order[0]] * 100
    color  = CLASS_COLORS[top]

    # Bars
    bars = ''
    for i in order:
        cls = labels[i]; pct = proba[i]*100; c = CLASS_COLORS[cls]
        bold = 'font-weight:700;' if i == order[0] else ''
        bars += f'''
        <div style="margin-bottom:10px;">
          <div style="display:flex;align-items:center;gap:10px;">
            <span style="width:145px;font-size:13px;{bold}color:{c};">{CLASS_ICONS[cls]} {cls}</span>
            <div style="flex:1;background:#e9ecef;border-radius:8px;height:22px;overflow:hidden;">
              <div style="width:{pct:.1f}%;background:{c};height:22px;border-radius:8px;"></div>
            </div>
            <span style="width:52px;text-align:right;font-size:13px;{bold}">{pct:.1f}%</span>
          </div>
        </div>'''

    badge_c = '#dc3545' if top_p>=80 else '#fd7e14' if top_p>=60 else '#6c757d'
    badge_t = 'HIGH CONFIDENCE' if top_p>=80 else 'MODERATE CONFIDENCE' if top_p>=60 else 'LOW CONFIDENCE'

    bmi_note = f' (calculated from {height_cm:.0f} cm / {weight_kg:.1f} kg)' \
               if bmi_mode=='Calculate from Height & Weight' else ''

    html = f'''
    <div style="font-family:'Segoe UI',Arial,sans-serif;max-width:720px;">
      <div style="border:2px solid {color};border-radius:14px;
                  background:linear-gradient(135deg,#fff 0%,#f8f9fa 100%);
                  padding:20px 24px;margin-bottom:14px;
                  box-shadow:0 4px 16px rgba(0,0,0,.08);">
        <div style="display:flex;align-items:center;gap:14px;margin-bottom:10px;">
          <span style="font-size:2.4rem;">{CLASS_ICONS[top]}</span>
          <div>
            <h2 style="margin:0;color:{color};font-size:1.6rem;">{top}</h2>
            <span style="background:{badge_c};color:#fff;font-size:11px;
                         font-weight:700;padding:2px 10px;border-radius:20px;">
              {badge_t} — {top_p:.1f}%
            </span>
          </div>
        </div>
        <p style="margin:0 0 6px;font-size:13px;color:#777;">
          BMI used: <strong>{bmi:.1f} kg/m²</strong>{bmi_note}
        </p>
        <p style="margin:0 0 18px;font-size:14px;color:#555;line-height:1.6;">
          {CLASS_DESC[top]}
        </p>
        <h4 style="margin:0 0 10px;font-size:13px;color:#333;">CLASS PROBABILITIES</h4>
        {bars}
      </div>
      <div style="background:#fff3cd;border:1px solid #ffc107;border-radius:8px;
                  padding:10px 14px;font-size:12px;color:#856404;line-height:1.5;">
        ⚠️ <strong>Educational use only.</strong> Trained on <em>synthetic</em> data.
        Does <strong>not</strong> constitute medical advice. Consult a qualified clinician.
      </div>
    </div>'''
    return html, f"{bmi:.1f}"

print("✅ Prediction function ready.")

## Step 5 — Build the Gradio Interface

In [ ]:
CSS = """
body,.gradio-container{font-family:'Segoe UI',Arial,sans-serif!important;}
#app-header{
  background:linear-gradient(135deg,#0f0c29,#302b63,#24243e);
  padding:26px 32px;border-radius:14px;margin-bottom:18px;
  box-shadow:0 6px 24px rgba(0,0,0,.35);}
#app-header h1{color:#e94560;margin:0 0 6px;font-size:2rem;}
#app-header p{color:#adb5bd;margin:0;font-size:.95rem;line-height:1.5;}
.panel{background:#f8f9fa;border-radius:12px;padding:16px 18px;
        border:1px solid #dee2e6;margin-bottom:10px;}
.bmi-box{background:#e8f4fd;border:1px solid #bee5f5;border-radius:10px;
          padding:14px 16px;margin-bottom:10px;}
#classify-btn{
  background:linear-gradient(90deg,#e94560,#c0392b)!important;
  color:#fff!important;font-size:1rem!important;
  border-radius:10px!important;padding:12px!important;
  box-shadow:0 4px 12px rgba(233,69,96,.35)!important;
  border:none!important;}
#classify-btn:hover{opacity:.9!important;}
footer{display:none!important;}
"""

with gr.Blocks(css=CSS, title="Cardiac Disease Classifier") as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div id="app-header">
      <h1>🫀 Cardiac Disease Classifier</h1>
      <p>Enter a patient's clinical values and click <strong>Classify Patient</strong>.
         BMI can be entered directly <em>or</em> calculated automatically from height and weight.<br>
         Classifies into:
         <span style='color:#27ae60;font-weight:600;'>Healthy</span> ·
         <span style='color:#e74c3c;font-weight:600;'>CAD</span> ·
         <span style='color:#f39c12;font-weight:600;'>Arrhythmia</span> ·
         <span style='color:#8e44ad;font-weight:600;'>Heart Failure</span>
      </p>
    </div>
    """)

    with gr.Row():

        # ── LEFT COLUMN ───────────────────────────────────────────────────────
        with gr.Column(scale=1):

            # BMI panel
            with gr.Group(elem_classes='bmi-box'):
                gr.Markdown("### 📐 BMI Calculator")
                bmi_mode = gr.Radio(
                    choices=['Calculate from Height & Weight', 'Enter BMI directly'],
                    value='Calculate from Height & Weight',
                    label='BMI Input Mode')
                with gr.Row():
                    height_cm  = gr.Number(value=170, label='Height (cm)',
                                           minimum=100, maximum=230)
                    weight_kg  = gr.Number(value=75,  label='Weight (kg)',
                                           minimum=20,  maximum=250)
                bmi_display = gr.Textbox(value='25.9', label='Calculated BMI (kg/m²)',
                                         interactive=False)
                bmi_slider  = gr.Slider(15, 55, value=26, step=0.1,
                                        label='BMI (kg/m²) — used when entering directly')

                # Live BMI update
                def update_bmi(h, w):
                    if h and w and h > 0 and w > 0:
                        return f"{w / (h/100)**2:.1f}"
                    return "—"
                height_cm.change(update_bmi, [height_cm, weight_kg], bmi_display)
                weight_kg.change(update_bmi, [height_cm, weight_kg], bmi_display)

            # Continuous measurements
            with gr.Group(elem_classes='panel'):
                gr.Markdown("### 📊 Continuous Measurements")
                age   = gr.Slider(20,  90,   value=55,  step=1,   label='Age (years)')
                sbp   = gr.Slider(80,  220,  value=130, step=1,   label='Systolic BP (mmHg)')
                ldl   = gr.Slider(40,  300,  value=130, step=1,   label='LDL Cholesterol (mg/dL)')
                gluc  = gr.Slider(40,  450,  value=100, step=1,   label='Fasting Glucose (mg/dL)')
                hr    = gr.Slider(20,  200,  value=75,  step=1,   label='Heart Rate (bpm)')
                spo2  = gr.Slider(70,  100,  value=97,  step=0.5, label='SpO₂ (%)')
                ef    = gr.Slider(10,  80,   value=60,  step=1,   label='Ejection Fraction (%)')
                bnp   = gr.Slider(1,   3000, value=80,  step=1,   label='BNP (pg/mL)')

        # ── RIGHT COLUMN ──────────────────────────────────────────────────────
        with gr.Column(scale=1):
            with gr.Group(elem_classes='panel'):
                gr.Markdown("### 🔢 Symptoms & Clinical Findings")

                sob = gr.Radio(
                    choices=[(0,'None'),(1,'Mild'),(2,'Moderate'),(3,'Severe')],
                    value=0, label='Shortness of Breath (SOB)')
                ct  = gr.Radio(
                    choices=[(0,'None'),(1,'Non-anginal'),(2,'Typical Angina')],
                    value=0, label='Chest Tightness')
                sm  = gr.Radio(
                    choices=[(0,'Never'),(1,'Light'),(2,'Moderate'),(3,'Heavy')],
                    value=0, label='Smoking Status')
                dm  = gr.Radio(
                    choices=[(0,'Normal'),(1,'Pre-DM'),(2,'Type 2'),(3,'Type 1')],
                    value=0, label='Diabetes Status')
                ed  = gr.Radio(
                    choices=[(0,'None'),(1,'Mild'),(2,'Severe')],
                    value=0, label='Leg Edema')
                pal = gr.Radio(
                    choices=[(0,'None'),(1,'Occasional'),(2,'Frequent')],
                    value=0, label='Palpitations')
                ecg_inp = gr.Radio(
                    choices=[(0,'Normal'),(1,'Tachy/Brady'),(2,'AF'),(3,'VT')],
                    value=0, label='ECG Heart Rhythm')

    # ── Button ────────────────────────────────────────────────────────────────
    btn = gr.Button('🔍  Classify Patient', elem_id='classify-btn')

    # ── Output ────────────────────────────────────────────────────────────────
    output     = gr.HTML()
    bmi_out    = gr.Textbox(visible=False)   # carries back calculated BMI

    # ── Examples ──────────────────────────────────────────────────────────────
    gr.Examples(
        label='📋 Example Patients — click a row to load',
        examples=[
            # ht   wt   bmi  mode                       age sbp  ldl  gluc hr  spo2 ef  bnp sob ct sm dm ed pal ecg
            [170, 67,  23.2,'Calculate from Height & Weight', 30, 115,  95,  88, 68, 98, 65,  35,  0, 0, 0, 0, 0,  0,  0],
            [175, 99,  32.3,'Calculate from Height & Weight', 65, 162, 210, 185, 84, 93, 46, 340,  2, 2, 3, 2, 1,  1,  2],
            [165, 84,  30.9,'Calculate from Height & Weight', 52, 147, 148, 125,118, 92, 55, 190,  2, 0, 1, 1, 0,  2,  2],
            [168, 87,  30.8,'Calculate from Height & Weight', 72, 148, 118, 265, 96, 89, 58, 950,  3, 1, 2, 2, 2,  1,  3],
        ],
        inputs=[height_cm, weight_kg, bmi_slider, bmi_mode,
                age, sbp, ldl, gluc, hr, spo2, ef, bnp,
                sob, ct, sm, dm, ed, pal, ecg_inp],
    )

    # ── Wiring ────────────────────────────────────────────────────────────────
    btn.click(
        fn=predict,
        inputs=[height_cm, weight_kg, bmi_slider, bmi_mode,
                age, sbp, ldl, gluc, hr, spo2, ef, bnp,
                sob, ct, sm, dm, ed, pal, ecg_inp],
        outputs=[output, bmi_out],
    )

print("✅ Interface built.")

## Step 6 — Launch the App

The cell below launches the Gradio app.

- **Local URL** — opens in your browser on this machine
- **Public URL** — a temporary `gradio.live` link anyone can use (valid ~72 h)

> On **Replit** the webview auto-detects port 7860 and shows the interface directly.
> On **Google Colab** click the public link printed below.
> On **Jupyter (local)** click the local URL.

In [ ]:
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,          # generates public gradio.live link
    show_error=True,
    quiet=False,
)